# Naveen — Final Epoch-150 Task 3 Metrics

Evaluation and recording only: epoch 150, global step 45000, seed 171. Existing predictions and official FID/MiFID evidence are preserved. No training loops or image exports.


## Paths and deterministic policy

Project-relative paths; all sorted Monet and first 300 sorted Photos, with same-stem existing translations. Isolated dependencies and metric-only pretrained weights are inside the new final-metrics folder.


In [ ]:
from pathlib import Path
import os, sys, random, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p/"src/models.py").is_file() and (p/"data/monet_jpg").is_dir())
FINAL_ROOT = ROOT/"outputs_repro/final_metrics_epoch150"
sys.path.insert(0,str(ROOT/"src"))
sys.path.insert(0,str(FINAL_ROOT/"packages_readable"))
os.environ["TORCH_HOME"] = str(FINAL_ROOT/"torch_cache")
os.environ["MPLCONFIGDIR"] = str(FINAL_ROOT/"runtime/matplotlib")
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
import numpy as np
import pandas as pd
import torch
import torchvision
from torch import nn
from torchvision import transforms as T
from PIL import Image
from tqdm import tqdm
from IPython.display import display
import final_epoch150_metrics as fm
SEED = 171
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
models = torchvision.models
print("Output:",fm.OUT_REL.as_posix(),"Device:",device,"Seed:",SEED,"Cap:",fm.CAP)


## Verify existing inputs, checkpoint and records

Verify every stored image count, filename and 256×256 RGB mode. Confirm epoch/step/seed, official CSV values, existing parameter-count output and complete 1–150 training history. Hash protected artifacts before metric computation.


In [ ]:
ctx = fm.prepare_evaluation(ROOT)
display(pd.DataFrame([ctx["official_scores"]]))
print("Checkpoint:",ctx["checkpoint"],"SHA256:",ctx["checkpoint_sha256"])
print("History epochs:",len(ctx["history"]),"Global step:",int(ctx["epoch150"]["global_step"]))


## Environment and GPU

Record exact package versions, CUDA/driver and evaluation-only model-weight hashes. Training VRAM is read separately from its existing CSV.


In [ ]:
environment = fm.evaluation_environment(ctx)
print(json.dumps(environment,indent=2))


## Course protocol search and reuse

The assignment lists metrics and defines content cosine as input versus translation, but supplies no additional algorithm or LPIPS pairing. Task-2 classifier precision/recall is not substituted. Use the requested fallback protocols and retain course-search evidence.


In [ ]:
reuse = {
 "searched":["src","reference","../../Downloads/DATA266-Lab1-Pair25-main.zip",
             "../../Downloads/DATA266_Lab1_Fall_2026.pdf","../task3_gpu/DATA266_Lab1_Fall_2026.pdf",
             "../../Downloads/CycleGAN_Training_Flow_Study_Guide.pdf"],
 "course_page":"DATA266_Lab1_Fall_2026.pdf, Task 3 metrics, page 9",
 "reused_exactly":{"feature_extraction":"Epoch-150 official instructor evaluator cell 5",
                  "generator":"src/models.py Generator",
                  "cycle_preprocessing":"src/dataset.py load_image(training=False)",
                  "parameters":"Existing executed step-18 notebook output",
                  "training":"Existing epochs CSV history"},
 "no_additional_course_algorithms_found":True,
 "LPIPS_pairing":"Real input vs cycle reconstruction; no paired target protocol supplied",
 "fallbacks":{"KID":"Unbiased degree-3 polynomial MMD",
              "precision_recall":"k=3 nearest-neighbor feature manifolds",
              "content_cosine":"Input vs same-filename existing translation"}}
fm.save_json(ctx,"manifests/protocol_search_and_reuse.json",reuse)
print(json.dumps(reuse,indent=2))
fm.metric_sanity_checks()


## Exact instructor Inception extraction

The following definitions are copied verbatim from instructor cell 5. Extract 300×2048 features per set once; reuse them for KID, generative precision/recall and content cosine. FID/MiFID are not recomputed.


In [ ]:
# Inception feature extractor
def get_inception_model():
    inception = models.inception_v3(
        weights=models.Inception_V3_Weights.IMAGENET1K_V1,
        transform_input=False
    )
    inception.fc = nn.Identity()
    inception.to(device)
    inception.eval()
    return inception

INCEPTION_TF = T.Compose([
    T.Resize(299),
    T.CenterCrop(299),
    T.ToTensor(),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

def load_batch(paths):
    imgs = []
    for p in paths:
        img = Image.open(p).convert("RGB")
        imgs.append(INCEPTION_TF(img))
    return torch.stack(imgs, dim=0)

@torch.no_grad()
def get_activations(model, image_paths, batch_size=32):
    feats = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc="Inception activations"):
        batch_paths = image_paths[i:i+batch_size]
        x = load_batch(batch_paths).to(device)
        f = model(x).detach().cpu().numpy()
        feats.append(f)
    return np.concatenate(feats, axis=0)
print("Exact instructor feature-extraction definitions loaded; no FID/MiFID computation.")


In [ ]:
inception = get_inception_model()
features = {}
for label, paths in ctx["paths"].items():
    print("Extracting features:",label,len(paths),flush=True)
    features[label] = get_activations(inception,[str(p) for p in paths],batch_size=32)
    assert features[label].shape == (300,2048) and np.isfinite(features[label]).all()
with fm.output_path(ctx,"inception_features.npz").open("xb") as feature_file:
    np.savez_compressed(feature_file,**features)
fm.save_json(ctx,"manifests/inception_feature_protocol.json",{
 "features":2048,"layer":"Pooled pre-classifier output; fc=Identity",
 "weights":"IMAGENET1K_V1","weights_sha256":ctx["inception_weights_sha256"],
 "preprocessing":repr(INCEPTION_TF),"samples_per_set":300,"batch_size":32,"seed":171,
 "reuse":"Same arrays for KID, precision/recall and content cosine; no stored FID feature cache available."})
print({k:v.shape for k,v in features.items()})


## KID — both directions

Unbiased polynomial MMD: kernel (x·y/2048+1)^3, exclude within-set diagonals, include all cross terms. 100 subsets of 100 without replacement, PCG64 seed 171 reset per direction. Raw mean and population std (ddof=0); no clipping or rescaling.


In [ ]:
kid = {}
for direction, real_key in [("A2B","real_B"),("B2A","real_A")]:
    kid[direction] = fm.kid_from_features(features[real_key],features["gen_"+direction])
fm.save_json(ctx,"kid_results.json",kid)
display(pd.DataFrame({d:{k:v[k] for k in ("mean","std","subsets","subset_size","seed")} for d,v in kid.items()}).T)


## Generative precision and recall

300 real/generated raw Inception features per direction. Closed Euclidean manifolds with radii to the third other nearest neighbor (k=3). Precision is generated-in-real membership; recall is real-in-generated membership.


In [ ]:
pr = {}
for direction,real_key in [("A2B","real_B"),("B2A","real_A")]:
    pr[direction] = fm.generative_precision_recall(features[real_key],features["gen_"+direction],k=3)
fm.save_json(ctx,"precision_recall_results.json",pr)
display(pd.DataFrame({d:{k:v[k] for k in ("precision","recall","k","real_samples","generated_samples")} for d,v in pr.items()}).T)


## Content-preservation cosine

Input versus its same-filename existing translation. Reuse instructor Inception features and report mean/population std over 300 pairs per direction.


In [ ]:
cosine = {}
for direction,source_key in [("A2B","real_A"),("B2A","real_B")]:
    assert [p.stem for p in ctx["paths"][source_key]] == [p.stem for p in ctx["paths"]["gen_"+direction]]
    cosine[direction] = fm.content_cosine(features[source_key],features["gen_"+direction])
fm.save_json(ctx,"content_cosine_results.json",cosine)
display(pd.DataFrame({d:{k:v[k] for k in ("mean","std")} for d,v in cosine.items()}).T)


## LPIPS reference model

Official lpips 0.1.4; default AlexNet backbone, calibrated version 0.1, eval/no_grad. Pair real input with cycle reconstruction in the same domain. Inputs are [-1,1], normalize=False. Record any dependency error without a proxy.


In [ ]:
from models import Generator as CycleGenerator
from dataset import load_image as cycle_load_image
lpips_model = None
try:
    import lpips
    lpips_model = lpips.LPIPS(net="alex",version="0.1",verbose=False).to(device).eval()
    lpips_model.requires_grad_(False)
    print("LPIPS",fm.importlib.metadata.version("lpips"),"AlexNet v0.1, eval/no_grad, 300/domain.")
    fm.save_json(ctx,"manifests/lpips_reference.json",{
      "package_version":fm.importlib.metadata.version("lpips"),"backbone":"alex","metric_version":"0.1",
      "pretrained":True,"pnet_rand":False,"normalize":False,"range":"[-1,1]","samples_per_domain":300,
      "std_ddof":0,"pairing":"real input vs cycle reconstruction",
      "calibration_sha256":fm.sha256(FINAL_ROOT/"packages_readable/lpips/weights/v0.1/alex.pth"),
      "backbone_weights_sha256":environment["AlexNet_weights_sha256"]})
except Exception as error:
    ctx["errors"].append({"metric":"LPIPS","error":repr(error)})
    fm.save_json(ctx,"manifests/lpips_unavailable.json",ctx["errors"][-1])
    print("LPIPS UNAVAILABLE:",repr(error),"; no substitution.")


## Cycle L1 and LPIPS — shared reconstruction pass

Use existing deterministic 256×256 RGB preprocessing, preserved epoch-150 generators, eval/no_grad and float32. A→B→A / B→A→B tensors stay in memory. No image export. Primary unweighted cycle L1 is on [0,1]; [-1,1] L1 is recorded separately. LPIPS uses this same pass.


In [ ]:
cycle = fm.cycle_and_lpips(ctx,CycleGenerator,cycle_load_image,lpips_model,device)
display(pd.DataFrame({k:v for k,v in cycle.items() if k.startswith("cycle_")}).T)
print("Cycle samples: 300/domain; no translation/reconstruction image files saved.")


## LPIPS results

Mean and population std (ddof=0) of per-image distances from the shared pass; 300 per domain.


In [ ]:
display(pd.DataFrame({d:cycle["LPIPS_"+d] for d in ("A","B")}).T)
print("Per-image evidence:",(fm.OUT_REL/"cycle_lpips_per_image.csv").as_posix())


## Existing training/stability and parameter counts

Read existing CSV and executed notebook outputs. Cycle/identity training losses retain weights 10/5. Time is exact logged elapsed_minutes; throughput and VRAM are epoch-150 training values.


In [ ]:
display(pd.DataFrame([ctx["epoch150"]]).T.rename(columns={0:"existing_epoch150_value"}))
display(pd.DataFrame.from_dict(ctx["parameter_counts"],orient="index",columns=["existing_parameter_count"]))
print("Total parameters:",sum(ctx["parameter_counts"].values()),"History:",len(ctx["history"]),"epochs")


## Loss/stability plots — epochs 1–150

Raw logged values without smoothing. Six standalone plot artifacts; prediction images are unchanged.


In [ ]:
plot_paths = fm.make_plots(ctx)
print("\n".join(plot_paths))


## Final metric tables

Official FID/MiFID remain unchanged. Human audit/agreement and Kaggle fields are literal PENDING.


In [ ]:
wide_table,long_table = fm.build_reports(ctx,kid,pr,cosine,cycle)
display(wide_table.T.rename(columns={0:"value"}))
display(long_table)
fm.document_protocol(ctx)
print("CSV evidence:",(fm.OUT_REL/"automated_metrics_epoch150.csv").as_posix(),
      (fm.OUT_REL/"full_metrics_report.csv").as_posix())


## Validation and hashes

Finite metrics, exact sample counts, readable artifacts, unchanged protected hashes. Notebook SHA256 is recorded after executed outputs are saved.


In [ ]:
fm.validate_outputs(ctx)
print("Executed evidence notebook:",fm.FINAL_NB_REL.as_posix())
print("Protocol:",(fm.OUT_REL/"evaluation_protocol.md").as_posix())
print("Errors:",ctx["errors"])
